# W_ref reference training on Kaggle

Trains the **retain-only reference** for one or more CIFAR-10 classes. A reference for
class *c* is a ResNet-18 trained on the 45,000 images that are **not** class *c*, so it
has genuinely never seen that class. It is the target the unlearning search aims at.

**Before running:** set *Accelerator* to **GPU (T4 or P100)** and *Internet* to **On**,
and attach the `medus-class-code` dataset. No weights dataset is needed - this notebook
trains from scratch.

**Do not edit any cell except the one marked EDIT THIS.** Everything else is checked
against the finished frog reference; changing a hyperparameter would make your model
incomparable with the others and the run would have to be thrown away.

Expect roughly **45-75 minutes per class** on a T4/P100.

## 0. EDIT THIS - which assignment file to run

This is the only line you change. Use the file you were given and nothing else.

In [ ]:
ASSIGNMENT = "trial_ship.yaml"   # <-- the only editable line in this notebook

# Others, for later. Do NOT switch to one of these until Yash confirms the ship
# trial passed local validation.
#   ASSIGNMENT = "classes_yash.yaml"      # airplane 0, automobile 1, bird 2
#   ASSIGNMENT = "classes_pragati.yaml"   # cat 3, deer 4, dog 5
#   ASSIGNMENT = "classes_aditya.yaml"    # horse 7, truck 9

TAG = ASSIGNMENT.replace("classes_", "").replace(".yaml", "")
print("assignment:", ASSIGNMENT)
print("output zip will be: reference_outputs_%s.zip" % TAG)

## 1. Copy the project into the writable working directory

`PROJECT_ROOT` is derived from the source file's location and every relative path in
every config resolves against it. `/kaggle/input` is read-only, so the first write to
`results/` would fail; the project is copied first.

In [ ]:
import os, shutil, subprocess, sys
from pathlib import Path

INPUT_CODE = Path("/kaggle/input/medus-class-code/MEDUS_Class_Unlearning")
PROJECT    = Path("/kaggle/working/MEDUS_Class_Unlearning")

assert INPUT_CODE.is_dir(), (
    f"code dataset not found at {INPUT_CODE}. Attach the 'medus-class-code' dataset. "
    f"Available: {sorted(p.name for p in Path('/kaggle/input').glob('*'))}"
)

if PROJECT.exists():
    shutil.rmtree(PROJECT)
shutil.copytree(INPUT_CODE, PROJECT)
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT / "src"))

ASSIGNMENT_PATH = PROJECT / "kaggle" / "reference_training" / ASSIGNMENT
assert ASSIGNMENT_PATH.is_file(), (
    f"assignment file {ASSIGNMENT} not found. Available: "
    f"{sorted(p.name for p in (PROJECT / 'kaggle' / 'reference_training').glob('*.yaml'))}"
)
print("project at", PROJECT)
print("assignment at", ASSIGNMENT_PATH)
print(ASSIGNMENT_PATH.read_text())

## 2. Dependencies

**torch is deliberately not installed.** Kaggle preinstalls a build matched to the
assigned accelerator; `pip install torch` resolves a different one and breaks GPU
access. `requirements.txt` omits it for that reason.

In [ ]:
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"],
    check=True,
)
print("dependencies installed (torch untouched)")

## 3. Verify CUDA

Asserted, not printed-and-hoped. Training on CPU would take well over a day and produce
the same file at the end, so a missing GPU must stop the notebook here.

In [ ]:
import torch

assert torch.cuda.is_available(), (
    "no GPU. Set Accelerator to GPU (T4 or P100) in the notebook settings and rerun."
)
print("torch", torch.__version__)
print("GPU  ", torch.cuda.get_device_name(0))

## 4. Safety check - DRY RUN, trains nothing

Prints the assigned person, the class id and name, the four split sizes, the seed, the
epoch count, the GPU, the output path and the checkpoint selection rule.

**Read the block it prints.** If the class is not the one you were assigned, or any split
size is not 5000 / 45000 / 1000 / 9000, stop here and tell Yash. The driver aborts on a
wrong split by itself, but a human should see it too.

In [ ]:
result = subprocess.run(
    [sys.executable, "kaggle/reference_training/train_references.py",
     "--assignment", str(ASSIGNMENT_PATH), "--dry-run"],
    cwd=str(PROJECT),
)
assert result.returncode == 0, "dry run failed - do not continue, send the log to Yash"
print()
print("DRY RUN OK - the plan above is what will be trained")

## 5. Train

200 epochs per class, SGD lr 0.1, momentum 0.9, weight decay 5e-4, step schedule x0.1
every 40 epochs, seed 42 - identical to the finished frog reference.

The best checkpoint is chosen **by `D_r_test` accuracy, with `D_r_test` loss as the
tie-breaker**. `D_f_test` is logged every epoch as a diagnostic and never influences
selection: a reference that classifies the forgotten class well is a *worse* reference,
not a better one.

This cell is the long one. Leave the tab open.

In [ ]:
result = subprocess.run(
    [sys.executable, "kaggle/reference_training/train_references.py",
     "--assignment", str(ASSIGNMENT_PATH)],
    cwd=str(PROJECT),
)
assert result.returncode == 0, "training failed - send the full log to Yash"
print()
print("TRAINING COMPLETE")

## 6. Package the outputs

Built from an allowlist: the `_best_dr` checkpoint, its JSON sidecar, the training log,
the summary, the environment snapshot, the split, and a manifest. CIFAR-10, caches and
the `_latest` / `_final` checkpoints are excluded.

In [ ]:
result = subprocess.run(
    [sys.executable, "kaggle/reference_training/package_outputs.py",
     "--tag", TAG, "--out-dir", "/kaggle/working"],
    cwd=str(PROJECT),
)
assert result.returncode == 0, "packaging failed"

zip_path = Path("/kaggle/working") / f"reference_outputs_{TAG}.zip"
assert zip_path.is_file(), f"expected {zip_path} to exist"
print()
print("DOWNLOAD THIS FILE:", zip_path.name,
      f"({zip_path.stat().st_size:,} bytes)")

## 7. Download and send back

1. Open the **Output** panel on the right of the Kaggle editor.
2. Download **`reference_outputs_<TAG>.zip`** (the name is printed above).
3. Send that zip to Yash. Do not rename it, do not unzip it, do not send anything else.

Yash validates it locally with `validate_reference_zip.py` before it is used for
anything. Until it passes, the class is not done.